# 00 — Création des données brutes (Parquet avec problèmes)

Le sujet demande de travailler sur des données brutes, avec des problèmes de qualité.
On part du CSV PaySim et on fabrique une source plus réaliste :

- des fichiers Parquet, un dossier par jour (`date=2024-01-01`, `date=2024-01-02`, ...) ;
- `step` est conservé (c'est le vrai temps de la simulation) ;
- `transaction_time` est une date d'affichage : `step - 1` heures après le 1er janvier 2024 à minuit ;
- des problèmes de qualité, introduits volontairement et documentés ci-dessous.

| # | Problème | Colonne | Proportion |
|---|---|---|---|
| 1 | Deux formats de date | `transaction_time` | 5 % |
| 2 | Code d'erreur 9999999999 (valeur aberrante) | `amount` | 0,1 % |
| 3 | Montants négatifs (erreur de signe) | `amount` | 0,5 % |
| 4 | Montant stocké en texte, virgule au lieu du point | `amount` | 5 % |
| 5 | Montant manquant | `amount` | 2 % |
| 6 | Libellés de type incohérents (minuscules, espaces, abréviations) | `type` | 14 % |
| 7 | Type manquant | `type` | 1 % |
| 8 | Soldes manquants | `oldbalanceOrg`, `newbalanceDest`, `oldbalanceDest` | 3 %, 3 %, 2 % |
| 9 | Label en texte : `0`/`1` ou `yes`/`no` | `isFraud` | 5 % en yes/no |
| 10 | Label manquant | `isFraud` | 0,1 % |
| 11 | Lignes en double | toutes | 1 % |

Le CSV Kaggle est déjà propre, donc on fabrique nous-mêmes une source brute pour pratiquer le nettoyage avec Spark. Comme on connaît les problèmes introduits, on peut vérifier que le notebook 01 les retrouve tous.

Prérequis : `data/source/PS_20174392719_1491204439457_log.csv`.
Ce notebook ne se lance qu'une seule fois.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("fraude-00-donnees-brutes")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "spark-master")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .config("spark.sql.shuffle.partitions", "18")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", "file:///tmp/spark-events")
         .getOrCreate())

DATA = "/opt/spark-data"
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 14:54:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
schema = ("step INT, type STRING, amount DOUBLE, "
          "nameOrig STRING, oldbalanceOrg DOUBLE, newbalanceOrig DOUBLE, "
          "nameDest STRING, oldbalanceDest DOUBLE, newbalanceDest DOUBLE, "
          "isFraud INT, isFlaggedFraud INT")

df = spark.read.csv(DATA + "/source/PS_20174392719_1491204439457_log.csv", header=True, schema=schema)
print("Lignes du CSV :", df.count())

[Stage 0:>                                                          (0 + 9) / 9]

Lignes du CSV : 6362620


6 362 620 transactions. Le schéma est écrit à la main : avec `inferSchema=True`, Spark relirait tout le fichier une deuxième fois pour deviner les types.

## Date d'affichage
`step` = 1 correspond à la première heure (00:00 - 01:00) du 1er janvier 2024.

In [3]:
brut = df.withColumn("transaction_time",
                     F.from_unixtime(F.unix_timestamp(F.lit("2024-01-01 00:00:00")) + (F.col("step") - 1) * 3600))
brut = brut.withColumn("date", F.substring("transaction_time", 1, 10))

## 1. Deux formats de date

In [4]:
brut = brut.withColumn("transaction_time",
                       F.when(F.rand(1) < 0.05,
                              F.date_format(F.to_timestamp("transaction_time"), "dd/MM/yyyy HH:mm"))
                        .otherwise(F.col("transaction_time")))

## 2 à 5. Problèmes sur le montant

In [5]:
# 2. code d'erreur
brut = brut.withColumn("amount", F.when(F.rand(2) < 0.001, 9999999999.0).otherwise(F.col("amount")))

# 3. erreur de signe
brut = brut.withColumn("amount", F.when(F.rand(3) < 0.005, -F.col("amount")).otherwise(F.col("amount")))

# 4. montant en texte, avec une virgule pour 5 % des lignes
brut = brut.withColumn("amount", F.format_string("%.2f", "amount"))
brut = brut.withColumn("amount", F.when(F.rand(4) < 0.05, F.regexp_replace("amount", "[.]", ","))
                                  .otherwise(F.col("amount")))

# 5. montant manquant
brut = brut.withColumn("amount", F.when(F.rand(5) < 0.02, F.lit(None)).otherwise(F.col("amount")))

## 6 et 7. Libellés de type incohérents et manquants

In [6]:
brut = brut.withColumn("alea", F.rand(6))
brut = brut.withColumn("type",
                       F.when(F.col("alea") < 0.01, F.lit(None))
                        .when(F.col("alea") < 0.08, F.lower("type"))
                        .when(F.col("alea") < 0.12, F.concat(F.lit(" "), F.col("type"), F.lit(" ")))
                        .when((F.col("alea") < 0.15) & (F.col("type") == "TRANSFER"), F.lit("TRF"))
                        .when((F.col("alea") < 0.15) & (F.col("type") == "CASH_OUT"), F.lit("CASHOUT"))
                        .otherwise(F.col("type")))

## 8. Soldes manquants

In [7]:
brut = brut.withColumn("oldbalanceOrg", F.when(F.rand(7) < 0.03, F.lit(None)).otherwise(F.col("oldbalanceOrg")))
brut = brut.withColumn("newbalanceDest", F.when(F.rand(8) < 0.03, F.lit(None)).otherwise(F.col("newbalanceDest")))
brut = brut.withColumn("oldbalanceDest", F.when(F.rand(9) < 0.02, F.lit(None)).otherwise(F.col("oldbalanceDest")))

## 9 et 10. Label en texte et label manquant

In [8]:
brut = brut.withColumn("alea", F.rand(10))
brut = brut.withColumn("isFraud",
                       F.when(F.col("alea") < 0.001, F.lit(None))
                        .when((F.col("alea") < 0.05) & (F.col("isFraud") == 1), F.lit("yes"))
                        .when(F.col("alea") < 0.05, F.lit("no"))
                        .otherwise(F.col("isFraud").cast("string")))
brut = brut.drop("alea")

## 11. Lignes en double, puis écriture en Parquet (un dossier par jour)

In [9]:
brut = brut.union(brut.sample(fraction=0.01, seed=11))

brut.write.mode("overwrite").partitionBy("date").parquet(DATA + "/brut/transactions")

## Vérification

In [10]:
verif = spark.read.parquet(DATA + "/brut/transactions")
print("Lignes :", verif.count())
print("Fichiers Parquet :", len(verif.inputFiles()))
verif.printSchema()
verif.show(10, truncate=False)

Lignes : 6426188
Fichiers Parquet : 78
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: string (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- transaction_time: string (nullable = true)
 |-- date: date (nullable = true)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+-------------------+----------+
|step|type    |amount  |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|transaction_time   |date      |
+----+--------+--------+-----------+-------------+--------------+-----------+------------

On obtient 6 426 188 lignes (6 362 620 + 63 568 doublons, soit 1 %) dans 78 fichiers Parquet, un dossier par jour.

On range en Parquet par jour parce que le format est compressé et rangé par colonne, et Spark peut ne lire que les jours dont il a besoin.

`amount` et `isFraud` sont maintenant du texte, comme dans une vraie source brute. L'aperçu montre déjà des problèmes : `payment` en minuscules, `11668,14` avec une virgule. `step = 1` donne bien `2024-01-01 00:00:00`.

In [11]:
spark.stop()